In [ ]:
import pypsa
import pandas as pd

import matplotlib.pyplot as plt
from _helpers import load_config

### Paths

In [ ]:
product = "steel"  # "eaf" or "steel" or "eaf-grid"
region = "Europe"  # e.g. "Europe"

In [ ]:
single_region = f"../../resources/lco-{product}/cost_year~2030/{region}/network_1-0partload.nc"
# single_region = f"../../resources/lcos/cost_year~2030/Europe/network_1.nc"

### Read product model

In [ ]:
n = pypsa.Network(single_region)

In [ ]:
n.statistics()

### Plots in hourly resolution

In [ ]:
n.statistics()

#### Electricity balance

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="electricity")

#### Electricity balance

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="hydrogen")

#### HBI balance

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="hot briquetted iron")

#### Steel balance

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="steel")

#### All carriers balance

In [ ]:
n.statistics.energy_balance.iplot.area() # Ely als negativen Stromverbraucher, H2 Verbrauch der DRI als linie

### Hourly plot (clean)

In [ ]:
df = n.statistics.energy_balance(aggregate_time=False).loc[
    :, 
    ["Photovoltaics", "Wind energy",
     "battery inverter (charging)", "battery inverter (discharging)",
     "electrolysis", "electric arc furnace", "direct reduction furnace"],
    "Electricity"
]

df = df.loc[:, pd.Timestamp("2013-07-23"):pd.Timestamp("2013-07-28")]

df = (
    df
    .reset_index(level=["component", "bus_carrier"], drop=True)
    .groupby("carrier")
    .sum()
)

In [ ]:
config_fn = "../../config/config.yaml"
colors = load_config(config_fn)["colors"]   

In [ ]:
show_carriers = [
    "electrolysis",
    "Photovoltaics",
    "Wind energy",
    "battery inverter (charging)",
    "battery inverter (discharging)",
    "electric arc furnace",
    "direct reduction furnace",
]

# transpose so time is on x-axis and divide by 4 to have smaller system
system_scale = 4

df_plot = df.loc[show_carriers].T.div(system_scale)

fig, ax = plt.subplots(figsize=(9, 4))

df_plot.plot.area(ax=ax, stacked=True, color=colors)

# Optional: overlay hydrogen consumption (as before)
n.statistics.energy_balance(aggregate_time=False).loc[
    "Link", "direct reduction furnace", "Hydrogen"
].loc[
    pd.Timestamp("2013-07-23"):pd.Timestamp("2013-07-28")
].div(system_scale).plot(
    ax=ax,
    color="black",
    linewidth=1.5,
    label="DRI H$_2$ consumption"
)

# Styling (matching your original)
ax.grid(True, alpha=0.5, linestyle="--")
ax.set_xlabel("Day", fontsize=9)
ax.set_ylabel("Electricity Balance in MW", fontsize=9)
ax.set_xlim(
    pd.Timestamp("2013-07-23"),
    pd.Timestamp("2013-07-28")
)

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)

plt.tight_layout()
plt.show()


### Part load behaviour

In [ ]:
n.statistics.capacity_factor.plot.bar()

In [ ]:
n.links.p_min_pu

No part load or minimum load requirement set in this case

### Energy balance

In [ ]:
n.statistics.energy_balance().div(1e3)

In [ ]:
# Electricity to steel ratio
steel_demand = n.statistics.energy_balance().loc[:, 'Steel', 'Steel'].values[0] # in t of steel
pv_supply = n.statistics.energy_balance().loc[:, 'Photovoltaics', 'Electricity'].values[0]  # in MWh
wind_supply = n.statistics.energy_balance().loc[:, 'Wind energy', 'Electricity'].values[0]  # in MWh

steel_demand
elec_to_steel = (pv_supply + wind_supply) * (-1) / steel_demand  # in MWh/t
elec_to_steel

In [ ]:
pv_supply

In [ ]:
wind_supply